In [1]:
# Correr línea solamente si aún no se han instalado los requerimientos. De lo contrario, activar entorno virtual y seguir con el script
# !pip install -r requirements.txt -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import shap
import joblib

d:\Anaconda\envs\EnvFlights\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import custom_class_copy as cc
import feature_engineering_functions as func

In [4]:
model = joblib.load('modelo_XGB_V2.1.joblib')

In [5]:
custom_model = cc.CustomPrediction(model)

In [6]:
data = [
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 09:12", 'distancia': 435},
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 13:12"},
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 19:12", 'distancia': 435},
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 20:12"},

    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 09:12", 'distancia': 812},
    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 13:12"},
    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 19:12", 'distancia': 812},
    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 00:12"},

    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 09:12"},
    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 13:12", 'distancia': 783},
    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 21:12", 'distancia': 783},
    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 00:12"},

    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 18:12", 'distancia': 1045},
    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 3:12"},
    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 20:12"},
    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 00:12", 'distancia': 1045}
]

In [7]:
# data = {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-11-30 09:12"}
# entrada = pd.DataFrame(data, index=[0])
# entrada["fecha_vuelo"] = pd.to_datetime(entrada["fecha_vuelo"])
# entrada

In [8]:
# entrada.info()

In [9]:
entrada = pd.DataFrame(data)
entrada["fecha_vuelo"] = pd.to_datetime(entrada["fecha_vuelo"])
entrada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   aerolinea           16 non-null     object        
 1   aeropuerto_origen   16 non-null     object        
 2   aeropuerto_destino  16 non-null     object        
 3   fecha_vuelo         16 non-null     datetime64[ns]
 4   distancia           8 non-null      float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 772.0+ bytes


In [10]:
r = custom_model.predict(entrada)

In [11]:
r[np.random.randint(0, len(r))]

{'Predicción': 'A tiempo',
 'Probabilidad de retraso': np.float64(42.88999938964844),
 'Distancia': np.float64(1045.0)}

In [12]:
for ind, element in enumerate(r):
  if ind % 4 == 0:
    print()
  print("Predicción: ", element['Predicción'], "     Probabilidad de retraso:", round(element['Probabilidad de retraso'],2), "%      Fecha:", entrada.loc[ind]['fecha_vuelo'])


Predicción:  A tiempo      Probabilidad de retraso: 25.53 %      Fecha: 2026-11-30 09:12:00
Predicción:  A tiempo      Probabilidad de retraso: 31.2 %      Fecha: 2026-11-30 13:12:00
Predicción:  A tiempo      Probabilidad de retraso: 36.39 %      Fecha: 2026-11-30 19:12:00
Predicción:  A tiempo      Probabilidad de retraso: 39.64 %      Fecha: 2026-11-30 20:12:00

Predicción:  A tiempo      Probabilidad de retraso: 45.42 %      Fecha: 2026-12-24 09:12:00
Predicción:  Retrasado      Probabilidad de retraso: 56.33 %      Fecha: 2026-12-24 13:12:00
Predicción:  Retrasado      Probabilidad de retraso: 72.28 %      Fecha: 2026-12-24 19:12:00
Predicción:  Retrasado      Probabilidad de retraso: 60.92 %      Fecha: 2026-12-24 00:12:00

Predicción:  A tiempo      Probabilidad de retraso: 44.98 %      Fecha: 2026-01-15 09:12:00
Predicción:  Retrasado      Probabilidad de retraso: 58.44 %      Fecha: 2026-01-15 13:12:00
Predicción:  Retrasado      Probabilidad de retraso: 66.25 %      Fecha: 2

In [13]:
# Ejemplo con explicación para una muestra aleatoria
muestra = entrada.sample(1)
tabla, parrafo = custom_model.explain(muestra)
prediccion = custom_model.predict(muestra)
print(parrafo)
print()
print("Predicción: ", prediccion[0]['Predicción'], "    Probabilidad de retraso:", round(prediccion[0]['Probabilidad de retraso'], 2), "%")
print()
print(muestra.to_dict(orient='records')[0])

Para esta predicción, el modelo tomó su decisión considerando principalmente: 
 aerolinea (28.92% de influencia con tendencia a aumentar la estimación de retraso), 
 hora de vuelo (21.12% de influencia con tendencia a aumentar la estimación de retraso), 
 y aeropuerto destino (15.45% de influencia con tendencia a reducir la estimación de retraso).
Las características restantes (5) agrupan el resto de las influencias (34.51%).

Predicción:  Retrasado     Probabilidad de retraso: 54.29 %

{'aerolinea': 'B6', 'aeropuerto_origen': 'FLL', 'aeropuerto_destino': 'BNA', 'fecha_vuelo': Timestamp('2026-01-15 00:12:00'), 'distancia': nan}
